# 梯度下降优化算法实现

梯度下降是深度学习中最基础的优化算法，本notebook演示SGD、Momentum、RMSprop和Adam等优化器的实现和对比。

## 1. 导入必要的库

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

## 2. 实现SGD优化器

最基础的优化算法：$\theta = \theta - \eta \nabla L(\theta)$

In [ ]:
class SGD:
    """随机梯度下降优化器"""
    
    def __init__(self, learning_rate=0.01):
        self.learning_rate = learning_rate
    
    def update(self, params, grads):
        """更新参数"""
        return params - self.learning_rate * grads

## 3. 实现Momentum优化器

在SGD基础上加入动量项，加速收敛并减少震荡。

In [ ]:
class Momentum:
    """动量优化器"""
    
    def __init__(self, learning_rate=0.01, momentum=0.9):
        self.learning_rate = learning_rate
        self.momentum = momentum
        self.velocity = None
    
    def update(self, params, grads):
        """使用动量更新参数"""
        if self.velocity is None:
            self.velocity = np.zeros_like(params)
        
        # 累积动量
        self.velocity = self.momentum * self.velocity + grads
        
        # 更新参数
        return params - self.learning_rate * self.velocity

## 4. 实现RMSprop优化器

自适应学习率方法，对每个参数使用不同的学习率。

In [ ]:
class RMSprop:
    """RMSprop优化器"""
    
    def __init__(self, learning_rate=0.01, decay_rate=0.9, epsilon=1e-8):
        self.learning_rate = learning_rate
        self.decay_rate = decay_rate
        self.epsilon = epsilon
        self.cache = None
    
    def update(self, params, grads):
        """使用RMSprop更新参数"""
        if self.cache is None:
            self.cache = np.zeros_like(params)
        
        # 更新梯度平方的移动平均
        self.cache = self.decay_rate * self.cache + (1 - self.decay_rate) * grads**2
        
        # 自适应学习率更新
        return params - self.learning_rate * grads / (np.sqrt(self.cache) + self.epsilon)

## 5. 实现Adam优化器

结合Momentum和RMSprop的优点，是目前最常用的优化器。

In [ ]:
class Adam:
    """Adam优化器"""
    
    def __init__(self, learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.learning_rate = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = None  # 一阶矩估计
        self.v = None  # 二阶矩估计
        self.t = 0     # 时间步
    
    def update(self, params, grads):
        """使用Adam更新参数"""
        if self.m is None:
            self.m = np.zeros_like(params)
            self.v = np.zeros_like(params)
        
        self.t += 1
        
        # 更新矩估计
        self.m = self.beta1 * self.m + (1 - self.beta1) * grads
        self.v = self.beta2 * self.v + (1 - self.beta2) * grads**2
        
        # 偏差修正
        m_hat = self.m / (1 - self.beta1**self.t)
        v_hat = self.v / (1 - self.beta2**self.t)
        
        # 更新参数
        return params - self.learning_rate * m_hat / (np.sqrt(v_hat) + self.epsilon)

## 6. 定义测试函数

我们使用两个经典的测试函数：
1. 简单二次函数：$f(x, y) = x^2 + y^2$
2. Rosenbrock函数：$f(x, y) = (1-x)^2 + 100(y-x^2)^2$

In [ ]:
# 简单二次函数
def quadratic_loss(params):
    """二次损失函数"""
    return params[0]**2 + params[1]**2

def quadratic_grad(params):
    """二次函数的梯度"""
    return 2 * params

# Rosenbrock函数（香蕉函数）
def rosenbrock_loss(params):
    """Rosenbrock函数"""
    x, y = params
    return (1 - x)**2 + 100 * (y - x**2)**2

def rosenbrock_grad(params):
    """Rosenbrock函数的梯度"""
    x, y = params
    dx = -2 * (1 - x) - 400 * x * (y - x**2)
    dy = 200 * (y - x**2)
    return np.array([dx, dy])

## 7. 优化过程可视化函数

In [ ]:
def visualize_optimization(optimizers, loss_fn, grad_fn, init_params, 
                          num_iterations=100, x_range=(-5, 5), y_range=(-5, 5)):
    """可视化不同优化器的优化过程"""
    
    fig, axes = plt.subplots(1, len(optimizers), figsize=(6*len(optimizers), 5))
    if len(optimizers) == 1:
        axes = [axes]
    
    # 创建网格用于绘制等高线
    x = np.linspace(x_range[0], x_range[1], 100)
    y = np.linspace(y_range[0], y_range[1], 100)
    X, Y = np.meshgrid(x, y)
    Z = np.zeros_like(X)
    
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Z[i, j] = loss_fn(np.array([X[i, j], Y[i, j]]))
    
    # 对每个优化器进行优化并可视化
    for idx, (name, optimizer) in enumerate(optimizers.items()):
        ax = axes[idx]
        
        # 绘制损失函数等高线
        contour = ax.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.6)
        ax.clabel(contour, inline=True, fontsize=8)
        
        # 优化过程
        params = init_params.copy()
        trajectory = [params.copy()]
        
        for _ in range(num_iterations):
            grads = grad_fn(params)
            params = optimizer.update(params, grads)
            trajectory.append(params.copy())
        
        # 绘制优化轨迹
        trajectory = np.array(trajectory)
        ax.plot(trajectory[:, 0], trajectory[:, 1], 'r.-', 
                linewidth=2, markersize=8, alpha=0.7, label='优化路径')
        ax.plot(trajectory[0, 0], trajectory[0, 1], 'go', markersize=12, label='起点')
        ax.plot(trajectory[-1, 0], trajectory[-1, 1], 'r*', markersize=15, label='终点')
        
        ax.set_xlabel('参数 x')
        ax.set_ylabel('参数 y')
        ax.set_title(f'{name} 优化过程')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 8. 收敛速度对比函数

In [ ]:
def compare_convergence(optimizers, loss_fn, grad_fn, init_params, num_iterations=100):
    """比较不同优化器的收敛速度"""
    
    plt.figure(figsize=(10, 6))
    
    for name, optimizer in optimizers.items():
        params = init_params.copy()
        losses = []
        
        for _ in range(num_iterations):
            losses.append(loss_fn(params))
            grads = grad_fn(params)
            params = optimizer.update(params, grads)
        
        plt.plot(losses, label=name, linewidth=2)
    
    plt.xlabel('迭代次数')
    plt.ylabel('损失值')
    plt.title('不同优化器的收敛速度对比')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.yscale('log')
    plt.show()

## 9. 测试简单二次函数

首先在简单的二次函数上测试各个优化器。

In [ ]:
# 初始化参数
init_params = np.array([4.0, 4.0])

# 创建优化器
optimizers = {
    'SGD': SGD(learning_rate=0.1),
    'Momentum': Momentum(learning_rate=0.1, momentum=0.9),
    'RMSprop': RMSprop(learning_rate=0.3, decay_rate=0.9),
    'Adam': Adam(learning_rate=0.3, beta1=0.9, beta2=0.999)
}

# 测试优化效果
print("简单二次函数优化结果：")
print("=" * 60)

for name, optimizer in optimizers.items():
    params = init_params.copy()
    print(f"\n{name}:")
    print(f"  初始参数: {params}, 损失: {quadratic_loss(params):.6f}")
    
    for _ in range(20):
        grads = quadratic_grad(params)
        params = optimizer.update(params, grads)
    
    print(f"  最终参数: {params}, 损失: {quadratic_loss(params):.6f}")

## 10. 可视化优化路径（二次函数）

In [ ]:
# 重新创建优化器（清除状态）
optimizers_viz = {
    'SGD': SGD(learning_rate=0.1),
    'Momentum': Momentum(learning_rate=0.1, momentum=0.9),
    'Adam': Adam(learning_rate=0.3, beta1=0.9, beta2=0.999)
}

visualize_optimization(optimizers_viz, quadratic_loss, quadratic_grad, 
                      init_params, num_iterations=50)

## 11. 收敛速度对比（二次函数）

In [ ]:
optimizers_conv = {
    'SGD': SGD(learning_rate=0.1),
    'Momentum': Momentum(learning_rate=0.1, momentum=0.9),
    'RMSprop': RMSprop(learning_rate=0.3, decay_rate=0.9),
    'Adam': Adam(learning_rate=0.3, beta1=0.9, beta2=0.999)
}

compare_convergence(optimizers_conv, quadratic_loss, quadratic_grad, 
                   init_params, num_iterations=50)

## 12. 测试Rosenbrock函数

Rosenbrock函数是一个更具挑战性的测试函数，具有狭长的峡谷形状。

In [ ]:
# 使用Rosenbrock函数测试
init_params_rb = np.array([-1.0, 2.0])

optimizers_rb = {
    'SGD': SGD(learning_rate=0.001),
    'Momentum': Momentum(learning_rate=0.001, momentum=0.9),
    'Adam': Adam(learning_rate=0.01, beta1=0.9, beta2=0.999)
}

visualize_optimization(optimizers_rb, rosenbrock_loss, rosenbrock_grad,
                      init_params_rb, num_iterations=200,
                      x_range=(-2, 2), y_range=(-1, 3))

## 13. Rosenbrock函数收敛对比

In [ ]:
compare_convergence(optimizers_rb, rosenbrock_loss, rosenbrock_grad,
                   init_params_rb, num_iterations=500)

## 14. 学习率对比实验

展示学习率对优化效果的影响。

In [ ]:
# 测试不同学习率
learning_rates = [0.01, 0.05, 0.1, 0.5]
init_params = np.array([4.0, 4.0])

plt.figure(figsize=(12, 8))

for i, lr in enumerate(learning_rates):
    plt.subplot(2, 2, i+1)
    
    optimizer = SGD(learning_rate=lr)
    params = init_params.copy()
    losses = []
    
    for _ in range(50):
        losses.append(quadratic_loss(params))
        grads = quadratic_grad(params)
        params = optimizer.update(params, grads)
    
    plt.plot(losses, linewidth=2)
    plt.xlabel('迭代次数')
    plt.ylabel('损失值')
    plt.title(f'学习率 = {lr}')
    plt.grid(True, alpha=0.3)
    plt.yscale('log')

plt.tight_layout()
plt.show()

## 15. 总结

通过本notebook我们实现并对比了四种常见的梯度下降优化算法：

1. **SGD**: 最简单但可能收敛慢
2. **Momentum**: 加速收敛，减少震荡
3. **RMSprop**: 自适应学习率
4. **Adam**: 结合Momentum和RMSprop的优点，是最常用的优化器

**关键发现**：
- Adam在大多数情况下表现最好
- 学习率的选择对优化效果影响很大
- Momentum可以帮助跳出局部最优
- 对于复杂的损失曲面（如Rosenbrock函数），自适应优化器优势明显